# QGAIN v4.1.0 — Standardized validation and figure completion

This notebook creates a **figure-only scientific supplement** from the immutable `qgain-v4.1.0` freeze. It does not decode audio, recompute feature values, modify the measurement freeze, construct a family scalar, or calibrate an accept/reject threshold.

Required outputs: completed family evaluation workbook, machine-readable checklist, standardized panels A–H, optional ML handoff panel J, source data, captions, provenance, and a candidate manifest ready for atomic sealing.

In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'MAIN outputs/02_FEATURE_REVIEWED').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('Could not locate the reviewed pipeline project root.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from paper1_qc_reviewed.qgain_figure_completion_v100 import build_package

FREEZE_ROOT = PROJECT_ROOT / 'MAIN outputs/02_FEATURE_REVIEWED' / '06_family_freezes' / 'gain_dynamics' / 'qgain-v4.1.0'
OUTPUT_ROOT = PROJECT_ROOT / 'MAIN outputs/02_FEATURE_REVIEWED/00_working_candidates' / 'gain_dynamics' / 'qgain-v4.1.0-figures-v1.0.0-candidate'
DOCS_ROOT = PROJECT_ROOT / 'docs reviewed'

print({'project_root': str(PROJECT_ROOT), 'freeze_root': str(FREEZE_ROOT), 'output_root': str(OUTPUT_ROOT)})

In [ ]:
freeze_manifest = json.loads((FREEZE_ROOT / 'manifests' / 'qgain_v410_freeze_manifest.json').read_text(encoding='utf-8'))
assert freeze_manifest['measurement_version'] == 'qgain-v4.1.0'
assert freeze_manifest['freeze_status'] == 'frozen'
assert freeze_manifest['recording_count'] == 519
assert freeze_manifest['participant_count'] == 224
print('Validated immutable source freeze.')
display(pd.DataFrame([freeze_manifest]).T.rename(columns={0: 'value'}).head(25))

In [ ]:
build_package(FREEZE_ROOT, OUTPUT_ROOT)

(OUTPUT_ROOT / 'docs').mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / 'tables').mkdir(parents=True, exist_ok=True)

for name in [
    'QGAIN_Family_Evaluation_Workbook_v1_0.docx',
    'QGAIN_Standardized_Validation_and_Figure_Package_README.md',
    'QGAIN_FIGURE_COMPLETION_CONTRACT_v1_0.md',
]:
    shutil.copy2(DOCS_ROOT / name, OUTPUT_ROOT / 'docs' / name)

for name in [
    'QGAIN_Validation_Checklist_v1_0.csv',
    'QGAIN_Ten_Domain_Dashboard_v1_0.csv',
    'QGAIN_Figure_Gallery_Index_v1_0.csv',
]:
    shutil.copy2(DOCS_ROOT / name, OUTPUT_ROOT / 'tables' / name)

print('Generated standardized QGAIN figure package and attached completed workbook/checklists.')

In [ ]:
figure_index = pd.read_csv(OUTPUT_ROOT / 'tables' / 'qgain_v410_figure_gallery_index.csv')
checklist = pd.read_csv(OUTPUT_ROOT / 'tables' / 'QGAIN_Validation_Checklist_v1_0.csv')
domain = pd.read_csv(OUTPUT_ROOT / 'tables' / 'QGAIN_Ten_Domain_Dashboard_v1_0.csv')
manifest = json.loads((OUTPUT_ROOT / 'manifests' / 'qgain_v410_figure_package_manifest.json').read_text(encoding='utf-8'))

assert len(figure_index) == 32
assert set('ABCDEFGH').issubset(set(figure_index['panel']))
assert figure_index['status'].eq('PASS').all()
assert manifest['feature_values_recomputed'] is False
assert manifest['required_panels_complete'] is True

display(domain)
display(figure_index.groupby(['panel', 'status']).size().rename('figure_count').reset_index())
display(pd.DataFrame([manifest]).T.rename(columns={0: 'value'}))

## Final local action

After this notebook finishes without errors:

1. Save the notebook.
2. Close JupyterLab.
3. Run `scripts/freeze_qgain_figure_package_v100.ps1`.

The seal script will rerun the package tests, preserve this executed notebook, generate a SHA-256 inventory, and atomically publish the immutable figure supplement under `MAIN outputs/02_FEATURE_REVIEWED/07_figure_packages/gain_dynamics/qgain-v4.1.0-figures-v1.0.0`.